## About Dataset


**enriched_df.csv** - raw data that is already preprocessed but still contains all original crime incidents; original data with mapped artifical offense ID
- 19,740 suspects (18,035 male and 1701 females)
- 7592 victims (4333 male and 3259 female victims)
- 11896 crimes (3199 index, 8697 non-index)
- 3678 focus crimes
- Min & max latitude is 15.113, 15.196
- Min & max longitude is 120.489, 120.639

## Final Dataset


**focus_df.csv** - only has crime incidents on focus crimes (both index and non-index); final data used for data visualizations and ML model
- 5067 suspects (4719 male, 346 female)
- 4040 victims (2509 male, 1531 female)
- 3678 focus crimes (3199 index, 479 non-index crime)
- Min & max latitude is 15.113, 15.169
- Min & max longitude is 120.488, 120.615

**Columns**
- **Offense ID** - ID of crime occurrence
- **Barangay** - Barangay of crime occurrence
- **Date** - date of crime occurrence
- **Time Committed** - time of crime occurrence
- **Offense Committed** - specific laws violated
- **Focus Crime** - e.g. 'Carnapping MC', 'Carnapping MV', 'Homicide', 'Murder', 'Physical Injuries', 'Rape', 'Robbery', 'Theft. Non-focus crime defaulted as 'Other'
- **Crime Type** - e.g. 'INDEX CRIME' or 'NON INDEX CRIME'
- **Case Status** - e.g. 'Case Solved', 'Cleared', 'Under Investigation'
- **Latitude** - vertical points;
- **Longitude** - horizontal points;
- **Victim Count** - count of victims per unique crimes;
- **Suspect Count** - count of suspects per unique crimes;
- **Year** - year extracted from 'Date' column
- **Month** - month extracted from 'Date' column
- **Day** - day extracted from 'Date' column
- **Day_of_Week** - name of day on 'Weekday' column.
- **Hour** - hour extracted from 'Date' column
- **Weekday** - day of week from 'Date' with format of 1 for Monday and 7 for Sunday
- **Week_of_Year** - week of year usually ranging from 1-52 weeks
- **Quarter** - quarter where date of crime belonged
- **Is_Weekend** - '1' if the crime occurred in weekend, else '0'
- **Time_of_Day** - 24 hours grouped into 6-hour bins (e.g. Midnight = 00:00-5:59,  Morning = 6:00-11:59, Afternoon = 12:00-17:59, Evening = 18:00-23:59)
- **Police Station** - police station where crime was reported or assigned to
- **Distance_from_Police** - distance of crime location from assigned police station in km
- **Nearest_Police_Station** - nearest police station even if not assigned
- **Nearest_Police_Distance** - nearest police station's distance from crime location in km
- **Num_Police_Stations_1km** - number of police stations within 1km radius on crime location
- **Num_Suspects** - number of suspects per crime
- **Avg_Suspects_Age** - average age of suspects per crime
- **Male_Suspects** - number of suspects who are male
- **Female_Suspects** - number of suspects who are female
- **Suspects_0_17** - suspects aged 17 years old or less
- **Suspects_18_25** - suspects aged between 18 and 25
- **Suspects_26_34** - suspects aged between 26 and 34
- **Suspects_35-44** - suspects aged between 35 and 44
- **Suspects_45-54** - suspects aged between 45 and 54
- **Suspects_55_64** - suspects aged between 55 and 64
- **Suspects_65_Above** - suspects aged 65 and above
- **Num_Victims** - number of victims per crime; there are 0 victims on some crimes especially non-index crimes
- **Avg_Victims_Age** - average age of victims per crime
- **Male_Victims** - number of victims who are male
- **Female_Victims** - number of victims who are female
- **Victims_0_17** - victims who are 17 years old or less;
- **Victims_18_25** - victims aged between 18 and 25
- **Victims_26_34** - victims aged between 26 and 34
- **Victims_35_44** - victims aged between 35 and 44
- **Victims_45_54** - victims aged between 45 and 54
- **Victims_55_64** -  victims aged between 55 and 64
- **Victims_65_Above** - victims aged 65 and above
- **Area_sqm** - area of barangay in sqm
- **Area_sqkm** - area of barangay in sqkm
- **Population_2020** - population of barangay on year 2020
- **Population_2024** - population of barangay on year 2024; used as latest population
- **Pop_Density_2020** - population density during 2020
- **Pop_Density_2024** - population density during 2024
- **Pop_Growth_Rate** - population growth rate comparison of 2020 and 2024 population


**Formula**

- **Population density** = # of people / area sqkm
- **Population growth rate** = ((2020 population/2024 population)*1/4)-1) * 100

**Others**
- **Crime Density** = crime count / area sqkm
- **Crime Rate per 1000** = (crime count / population) * 1000

## Import libraries

In [ ]:
# pip install pandas seaborn numpy matplotlib

import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_colwidth', None)
np.set_printoptions(linewidth=200)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.expand_frame_repr', False)

In [ ]:
# Load the dataset

crimes = pd.read_csv('Dataset/raw/crimes.csv')
suspects = pd.read_csv('Dataset/raw/suspects.csv')
victims = pd.read_csv('Dataset/raw/victims.csv')
angeles_city = pd.read_csv('Dataset/raw/angeles_city_other_info.csv')

## Crimes dataset

In [ ]:
crimes.head()

In [ ]:
crimes.info()

In [ ]:
crimes.shape

In [ ]:
crimes.columns

#### Modify/create columns

In [ ]:
# Make Date a datetime object
# Get weekday (1=Monday, 7=Sunday)
# Create column for day of the week, week of year, quarter, Is_Weekend

crimes['Date'] = pd.to_datetime(crimes['Date'])
crimes['Weekday'] = crimes['Date'].dt.isocalendar().day
crimes['Day_of_Week'] = crimes['Date'].dt.day_name()
crimes['Week_of_Year'] = crimes['Date'].dt.isocalendar().week
crimes['Quarter'] = crimes['Date'].dt.quarter
crimes['Is_Weekend'] = crimes['Day_of_Week'].isin(['Saturday', 'Sunday']).astype(int)


# Based on 24-hour format, create time of day bins
# Midnight (00:00-05:59), Morning (06:00-11:59), Afternoon (12:00-17:59),Evening (18:00-23:59)
hour_bins = [0, 6, 12, 18, 24]
hour_labels = ['Midnight', 'Morning', 'Afternoon', 'Evening']

crimes['Time_of_Day'] = pd.cut(crimes['Hour'],
                                    bins=hour_bins,
                                    labels=hour_labels,
                                    right=False) # right=False means 6 is in 'Morning'

crimes.head()

In [ ]:
## Specify focus crimes on last column

focus_crime = ['Murder', 'Robbery', 'Rape', 'Physical Injuries', 'Homicide', 'Theft', 'Carnapping MC', 'Carnapping MV']

murder = "MURDER|PARRICIDE"
homicide = "^HOMICIDE|ATTEMPTED HOMICIDE|FRUSTRATED HOMICIDE"
rape = "RAPE|ANTI-RAPE LAW|11648|RAPE WITH"
physical_injuries = "PHYSICAL INJURIES|SERIOUS PHYSICAL INJURIES|LESS SERIOUS PHYSICAL INJURIES|SLIGHT PHYSICAL INJURIES"
robbery = "ROBBERY|^ROBBERY WITH"
theft = "THEFT|QUALIFIED THEFT"
carnapping_mc = "CARNAPPING MC"
carnapping_mv = "CARNAPPING MV"

conditions = [
    crimes['Offense Committed'].str.contains(murder, case=False, na=False),
    crimes['Offense Committed'].str.contains(robbery, case=False, na=False),
    crimes['Offense Committed'].str.contains(rape, case=False, na=False),
    crimes['Offense Committed'].str.contains(physical_injuries, case=False, na=False),
    crimes['Offense Committed'].str.contains(homicide, case=False, na=False),
    crimes['Offense Committed'].str.contains(theft, case=False, na=False),
    crimes['Offense Committed'].str.contains(carnapping_mc, case=False, na=False),
    crimes['Offense Committed'].str.contains(carnapping_mv, case=False, na=False)
]

crimes['Focus_Crime'] = np.select(conditions, focus_crime, default='Other')


In [ ]:
crimes.isna().sum()

#### Data cleaning/imputing

Handling missing values:

- For 'Time Committed', MODE of both 'Barangay' and 'Offense Committed', or 'Barangay'.
- For 'Latitude' and 'Longitude', fill with MEAN of both 'Barangay' and 'Offense Committed', or 'Barangay'.
- For 'Case Solved Type', dropped column

In [ ]:
# Fill missing values in 'Time Committed' with MODE of same 'Barangay' and 'Offense Committed' or 'Barangay' only

crimes['Time Committed'] = crimes.groupby(['Barangay', 'Offense Committed'])['Time Committed'].transform(lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else np.nan))

if crimes['Time Committed'].isna().sum() > 0:
    crimes['Time Committed'] = crimes['Time Committed'].fillna(crimes.groupby('Barangay')['Time Committed'].transform(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan))

# Fill the missing values in latitude and longitude with the mean of the grouped column of 'Barangay' and 'Offense Committed'

crimes['Latitude'] = crimes['Latitude'].fillna(
    crimes.groupby(['Barangay', 'Offense Committed'])['Latitude'].transform('mean')
)

if crimes['Latitude'].isna().sum() > 0:
    crimes['Latitude'] = crimes['Latitude'].fillna(
        crimes.groupby('Barangay')['Latitude'].transform('mean')
    )

crimes['Longitude'] = crimes['Longitude'].fillna(
    crimes.groupby(['Barangay', 'Offense Committed'])['Longitude'].transform('mean'))

if crimes['Longitude'].isna().sum() > 0:
    crimes['Longitude'] = crimes['Longitude'].fillna(
        crimes.groupby('Barangay')['Longitude'].transform('mean')
    )

In [ ]:
## Clean and trim offense ID column for excess space or characters for this int object
crimes['Offense ID'] = crimes['Offense ID'].astype(str)
crimes['Offense ID'] = crimes['Offense ID'].str.replace(" ","")

In [ ]:
crimes.drop(columns=['Case Solved Type'], inplace=True)

In [ ]:
crimes.isna().sum()

#### Adding police station

In [ ]:
# Coordinates of Police Station 1-6 in Angeles City, Pampanga from Google Maps

police_station = {
    "Station": ["Police Station 1", "Police Station 2", "Police Station 3", "Police Station 4", "Police Station 5", "Police Station 6"],
    "Latitude": [15.1350, 15.1614, 15.1607, 15.1600, 15.1446, 15.15884],
    "Longitude": [120.5910, 120.6087, 120.6089, 120.5918, 120.5572, 120.59226]
}

In [ ]:
# Distance calculation from the assigned police station to the crime in KM

from geopy.distance import geodesic

# Create a mapping of police station names to their coordinates
station_coords = {}
for i in range(len(police_station['Station'])):
    station_coords[police_station['Station'][i]] = (police_station['Latitude'][i], police_station['Longitude'][i])

# Calculate distance to the assigned police station
def get_distance_to_station(row, assigned_station):
    if assigned_station in station_coords:
        crime_coords = (row['Latitude'], row['Longitude'])
        station_coord = station_coords[assigned_station]
        return geodesic(crime_coords, station_coord).meters / 1000  # Convert to kilometers
    return None

# Add the distance column
crimes['Distance_from_Police'] = crimes.apply(
    lambda row: get_distance_to_station(row, row['Police Station']),
    axis=1
)

In [ ]:
# What is the nearest police station (even if not assigned) and its distance in KM

def get_nearest_station(row):
    crime_coords = (row['Latitude'], row['Longitude'])
    nearest_station = None
    min_distance = float('inf')

    for station, coords in station_coords.items():
        distance = geodesic(crime_coords, coords).meters / 1000  # Convert to kilometers
        if distance < min_distance:
            min_distance = distance
            nearest_station = station

    return pd.Series([nearest_station, min_distance])
crimes[['Nearest_Police_Station', 'Nearest_Police_Distance']] = crimes.apply(get_nearest_station, axis=1)


In [ ]:
# What is the number of police stations within 1 km radius of the crime location
# If more police stations but lesser crime, then police presence deters crime

def count_nearby_stations(row, radius_km=1):
    crime_coords = (row['Latitude'], row['Longitude'])
    count = 0

    for coords in station_coords.values():
        distance = geodesic(crime_coords, coords).meters / 1000  # Convert to kilometers
        if distance <= radius_km:
            count += 1

    return count

crimes['Num_Police_Stations_1km'] = crimes.apply(count_nearby_stations, axis=1)

In [ ]:
crimes = crimes[['Offense ID', 'Barangay', 'Date', 'Time Committed', 'Offense Committed', 'Focus_Crime', 'Crime Type', 'Case Status', 'Latitude', 'Longitude', 'Victim Count', 'Suspect Count', 'Year', 'Month', 'Day','Day_of_Week','Hour', 'Weekday', 'Week_of_Year', 'Quarter', 'Is_Weekend', 'Time_of_Day', 'Police Station', 'Distance_from_Police', 'Nearest_Police_Station', 'Nearest_Police_Distance', 'Num_Police_Stations_1km']]
crimes.head()

In [ ]:
crimes.info()

## Suspects dataset

In [ ]:
suspects.head(3)

In [ ]:
suspects.shape

In [ ]:
suspects.info()

In [ ]:
suspects.isna().sum()

#### Data cleaning/imputation

Handling missing values:
- For 'Age', mean of BOTH 'Barangay' and 'Offense (Consolidated)', or 'Barangay'
- For 'Gender', mode of BOTH 'Barangay' and 'Offense (Consolidated'), else, 'Unknown'
- For 'Nationality' and 'Civil Status', it will be dropped due to many N/A

In [ ]:
# Fill suspect aged 0 with NaN
# Assumption: Age 0 is invalid and should be treated as missing value

suspects['Age'] = suspects['Age'].replace(0, np.nan)

# Fill missing values in 'Age' by mean grouped by 'Barangay' and 'Offense (Consolidated)'
suspects['Age'] = suspects['Age'].fillna(suspects.groupby(['Barangay', 'Offense (Consolidated)'])['Age'].transform(lambda x: x.fillna((x.mean()))))

if suspects['Age'].isna().sum() > 0:
    suspects['Age'] = suspects['Age'].fillna(suspects.groupby('Barangay')['Age'].transform('mean'))

# Fill missing values in Gender with the mode grouped by 'Barangay' and 'Offense (Consolidated)'

suspects['Gender'] = suspects['Gender'].fillna(suspects.groupby(['Barangay', 'Offense (Consolidated)'])['Gender'].transform(lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else 'Unknown')))
suspects['Gender'].isna().sum()

# 'Nationality' and 'Civil Status' will be dropped due to many missing values

suspects.drop(columns=['Nationality', 'Civil Status'], inplace=True)


#### Create/modify columns

In [ ]:
# Suspects Age Groups

age_bins = [0, 17, 25, 34, 44, 54, 64, np.inf]
age_labels = ['0-17', '18-25', '26-34', '35-44', '45-54', '55-64', '65+']

suspects['Age_Group'] = pd.cut(suspects['Age'],
                                   bins=age_bins,
                                   labels=age_labels,
                                   right=True) # right=True means 17 is in '0-17'

suspects.head()

In [ ]:
suspects['Offense ID'] = suspects['Offense ID'].astype(str)
suspects['Offense ID'] = suspects['Offense ID'].str.replace(" ","")

In [ ]:
suspects.isna().sum()

#### Grouping suspects based on 'Offense ID', 'Age', 'Gender', 'Age_Group'

- suspects_df for merging

In [ ]:
suspects = suspects[['Offense ID', 'Age', 'Gender','Age_Group']]

In [ ]:
suspects.shape

In [ ]:
suspects.head()

In [ ]:
suspects.isna().sum()

In [ ]:
## Aggregate suspects_summary based on 'Offense ID' since multiple suspects can be linked to a single offense from crimes dataset

suspects = suspects.groupby('Offense ID').agg(
    Num_Suspects =('Age', 'count'),
    Avg_Suspects_Age=('Age', 'mean'),
    Male_Suspects=('Gender', lambda x: (x == 'Male').sum()),
    Female_Suspects=('Gender', lambda x: (x == 'Female').sum()),
    Suspects_0_17=('Age_Group', lambda x: (x == '0-17').sum()),
    Suspects_18_25=('Age_Group', lambda x: (x == '18-25').sum()),
    Suspects_26_34=('Age_Group', lambda x: (x == '26-34').sum()),
    Suspects_35_44=('Age_Group', lambda x: (x == '35-44').sum()),
    Suspects_45_54=('Age_Group', lambda x: (x == '45-54').sum()),
    Suspects_55_64=('Age_Group', lambda x: (x == '55-64').sum()),
    Suspects_65_Above=('Age_Group', lambda x: (x == '65+').sum()),
).reset_index()

suspects.head()

In [ ]:
suspects.isna().sum()

## Victims dataset

In [ ]:
victims.head(3)

In [ ]:
victims.info()

In [ ]:
victims.shape

In [ ]:
victims.isna().sum()

#### Data handling/imputing

- For 'Age', mean of BOTH 'Barangay' and 'Offense (Consolidated)', or 'Barangay'
- For 'Gender', mode of BOTH 'Barangay' and 'Offense (Consolidated)'
- For 'Nationality' and 'Civil Status', dropped columns

In [ ]:
# Mean of 'Barangay' and 'Offense (Consolidated)', or 'Barangay' for 'Age'
# Mode of 'Barangay' and 'Offense (Consolidated)' for 'Gender'

victims['Age'] = victims['Age'].fillna(victims.groupby(['Barangay', 'Offense (Consolidated)'])['Age'].transform(lambda x: x.fillna((x.mean()))))

if victims['Age'].isna().sum() > 0:
    victims['Age'] = victims['Age'].fillna(victims.groupby('Barangay')['Age'].transform('mean'))


victims['Gender'] = victims.groupby(['Barangay', 'Offense (Consolidated)'])['Gender'].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'Unknown'))

In [ ]:
# Victims Age Group

age_bins = [0, 17, 25, 34, 44, 54, 64, np.inf]
age_labels = ['0-17', '18-25', '26-34', '35-44', '45-54', '55-64', '65+']

victims['Age_Group'] = pd.cut(victims['Age'],
                                bins=age_bins,
                                labels=age_labels,
                                right=True,
                                include_lowest=True) # right=True means 17 is in '0-17'
victims.head()

In [ ]:
victims.drop(columns=['Nationality', 'Civil Status'], inplace=True)

In [ ]:
victims.isna().sum()

In [ ]:
victims['Offense ID'] = victims['Offense ID'].astype(str)
victims['Offense ID'] = victims['Offense ID'].str.replace(" ","")

#### Grouping victims based on 'Offense ID', 'Age', 'Gender', 'Age Group'

In [ ]:
victims = victims[['Offense ID', 'Age', 'Gender', 'Age_Group']]

In [ ]:
## Aggregate victims_summary based on 'Offense ID' since multiple victims can be linked to a single offense from crimes dataset

victims = victims.groupby('Offense ID').agg(
    Num_Victims=('Age', 'count'),
    Avg_Victims_Age=('Age', 'mean'),
    Male_Victims=('Gender', lambda x: (x == 'Male').sum()),
    Female_Victims=('Gender', lambda x: (x == 'Female').sum()),
    Victims_0_17=('Age_Group', lambda x: (x == '0-17').sum()),
    Victims_18_25=('Age_Group', lambda x: (x == '18-25').sum()),
    Victims_26_34=('Age_Group', lambda x: (x == '26-34').sum()),
    Victims_35_44=('Age_Group', lambda x: (x == '35-44').sum()),
    Victims_45_54=('Age_Group', lambda x: (x == '45-54').sum()),
    Victims_55_64=('Age_Group', lambda x: (x == '55-64').sum()),
    Victims_65_Above=('Age_Group', lambda x: (x == '65+').sum()),
).reset_index()

victims.head()

## Demographics dataset

In [ ]:
angeles_city.head(3)

## Check observations of datasets

\- On original data, there are 11,896 crimes, 7592 victims, 19,740 suspects, and 33 barangays

In [ ]:
# Check if no data has been lost

print('Crimes:', crimes['Offense ID'].count())
print('\nVictims:', victims['Num_Victims'].sum())
print('\nSuspects:', suspects['Num_Suspects'].sum())
print('\nAngeles City Barangay:', angeles_city.shape[0])

## Merging data
- creation of enriched_df where crimes.csv, suspects.csv, victims.csv, angeles_city.csv, are comprehensively joined together by left-join by matching 'Offense ID' column
- crimes.csv will be the main data. suspects.csv will do left-join, then victims.csv, and lastly, angeles_city.
- crimes.csv became the left table and the rest are right table who will join the former
- with left join, left table will be the main data (crime.csv). if the right tables do not have matching 'Offense ID' on left table, the left table (crimes.csv) has still complete rows of data but the right table without match will just return blank values

**Merging Flow**

enriched_df = crimes <- suspects <- victims <- angeles_city

In [ ]:
enriched_df = crimes.merge(suspects, on='Offense ID', how='left') \
                        .merge(victims, on='Offense ID', how='left') \
                        .merge(angeles_city, on='Barangay', how='left')

In [ ]:
# Fill NaN values in suspects and victims summary columns with 0
# NaN values means crimes had no suspects or victims

col = ['Num_Suspects', 'Avg_Suspects_Age', 'Male_Suspects', 'Female_Suspects',
                'Suspects_0_17', 'Suspects_18_25', 'Suspects_26_34', 'Suspects_35_44',
                'Suspects_45_54', 'Suspects_55_64', 'Suspects_65_Above',
                'Num_Victims', 'Avg_Victims_Age', 'Male_Victims', 'Female_Victims', 'Victims_0_17',
                'Victims_18_25', 'Victims_26_34', 'Victims_35_44', 'Victims_45_54', 'Victims_55_64',
                'Victims_65_Above']

enriched_df[col] = enriched_df[col].fillna(0)

enriched_df.isna().sum()

In [ ]:
enriched_df_csv = enriched_df.to_csv('Dataset/cleaned/enriched_df.csv', index=False)

focus_df = enriched_df[enriched_df['Focus_Crime'] != 'Other']
focus_df_csv = focus_df.to_csv('Dataset/cleaned/focus_df.csv', index=False)


## Import Libraries

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Random Forest Classifier
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

## XGBoost Classifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import xgboost as xgb
# SMOTE
from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline


## Import Dataset

In [ ]:
focus_df = pd.read_csv('Dataset/cleaned/focus_df.csv', parse_dates=['Date'])

## About Dataset

In [ ]:
focus_df.head(3)

In [ ]:
focus_df.shape

In [ ]:
focus_df.info()

In [ ]:
focus_df.isna().sum()

In [ ]:
focus_df.describe()

In [ ]:
train_df = focus_df[(focus_df['Date'] < '2025-01-01')]
test_df = focus_df[(focus_df['Date'] >= '2025-01-01')]

In [ ]:
train_df.shape, test_df.shape

#### Grouping and aggregating columns

In [ ]:
def aggregate(df):
    agg_df = df.groupby(['Barangay', 'Month', 'Weekday', 'Time_of_Day']).agg(
        Crime_Count=('Offense ID', 'count'),

        # Temporal features (take mode or first for consistency)
        Avg_Hour = ('Hour', 'mean'),
        Mode_Hour=('Hour', lambda x: x.mode()[0] if not x.mode().empty else x.iloc[0]),
        Weekend_Crimes=('Is_Weekend', 'sum'),
        Weekday_Crimes=('Is_Weekend', lambda x: (~x.astype(bool)).sum()),

        # Spatial/demographic (constant per barangay, use first)
        Population=('Population_2024', 'first'),
        Pop_Density=('Pop_Density_2024', 'first'),
        Area_sqkm=('Area_sqkm', 'first'),

        # Police presence
        Avg_Distance_Police=('Distance_from_Police', 'mean'),
        Avg_Num_Stations_1km=('Num_Police_Stations_1km', 'mean'),

        # Crime characteristics
        Avg_Victims=('Num_Victims', 'median'),
        Avg_Suspects=('Num_Suspects', 'median'),

        # Focus crime distribution
        Murder_Count=('Focus_Crime', lambda x: (x == 'Murder').sum()),
        Theft_Count=('Focus_Crime', lambda x: (x == 'Theft').sum()),
        Robbery_Count=('Focus_Crime', lambda x: (x == 'Robbery').sum()),
        Physical_Injuries_Count=('Focus_Crime', lambda x: (x == 'Physical Injuries').sum()),
        Rape_Count=('Focus_Crime', lambda x: (x == 'Rape').sum()),
        Homicide_Count=('Focus_Crime', lambda x: (x == 'Homicide').sum()),
        Carnapping_MC_Count=('Focus_Crime', lambda x: (x == 'Carnapping MC').sum()),
        Carnapping_MV_Count=('Focus_Crime', lambda x: (x == 'Carnapping MV').sum())
    ).reset_index()

    agg_df['Crime_Rate_per_1000'] = (agg_df['Crime_Count'] / agg_df['Population']) * 1000
    agg_df['Crime_Density_sqkm'] = agg_df['Crime_Count'] / agg_df['Area_sqkm']
    agg_df['Weekend_Ratio'] = agg_df['Weekend_Crimes'] / (agg_df['Crime_Count'] + 1e-6)

    agg_df = agg_df.sort_values(['Barangay', 'Month', 'Weekday', 'Time_of_Day'])

    print(f"Aggregated dataset shape: {agg_df.shape}")

    return agg_df

In [ ]:
train_df = aggregate(train_df)
test_df = aggregate(test_df)

In [ ]:
train_df.head(2)

#### Applying Alarm Level
Applying alarm level on new dataset (new_df) based on thresholds from trained dataset (train_df)

In [ ]:
def classify_alarm(new_df, train_df):

    q25_baseline = train_df['Crime_Count'].quantile(0.25)
    q75_baseline = train_df['Crime_Count'].quantile(0.75)

    print(f"BASELINE THRESHOLDS (from 2017-2024 data):")
    print(f"   25th Percentile (Low/Medium boundary): {q25_baseline}")
    print(f"   75th Percentile (Medium/High boundary): {q75_baseline}")
    print(f" Mean of Crime Count: {train_df['Crime_Count'].mean().round(2)}")

    # Define a UNIVERSAL classification function using baseline thresholds
    def classify_alarm(count, q25, q75):
        if count <= q25:
            return 'Low'
        elif count <= q75:
            return 'Medium'
        else:
            return 'High'

    # Apply to 2017-2024 data
    train_df['Alarm_Level'] = train_df['Crime_Count'].apply(
    lambda x: classify_alarm(x, q25_baseline, q75_baseline)
)

    # Apply to 2025 data using THE SAME THRESHOLDS
    new_df['Alarm_Level'] = new_df['Crime_Count'].apply(
    lambda x: classify_alarm(x, q25_baseline, q75_baseline)
)
    # Verify distributions
    print("\n2017-2024 Alarm Level Distribution:")
    print(train_df['Alarm_Level'].value_counts().sort_index())

    print("\n2025 Alarm Level Distribution:")
    print(new_df['Alarm_Level'].value_counts().sort_index())

    print("\nNote: Both datasets now use the same crime alarm thresholds from 2017-2024 baseline!")


In [ ]:
classify_alarm(test_df, train_df)

## Correlation

#### Correlation of enriched_df

In [ ]:
enriched_df = pd.read_csv('Dataset/cleaned/enriched_df.csv', parse_dates=['Date'])
enriched_df = aggregate(enriched_df)

enr_cols = [
    'Crime_Count',    # Temporal
    'Avg_Hour', 'Weekend_Ratio',  # Time patterns
    'Population', 'Mode_Hour', 'Pop_Density', 'Area_sqkm',  # Demographic/spatial
     'Avg_Num_Stations_1km',  # Police presence
    'Avg_Victims', 'Avg_Suspects',  # Crime characteristics
    'Crime_Rate_per_1000', 'Crime_Density_sqkm',  # Derived metrics
]

corr_matrix_enr = enriched_df[enr_cols].corr(numeric_only=True)

plt.figure(figsize=(15, 12))
sns.heatmap(corr_matrix_enr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix - Enriched Dataset')
plt.show()


crime_corr_enr = corr_matrix_enr['Crime_Count'].sort_values(ascending=False)
print("Top correlations with Crime_Count (Enriched Dataset):")
print(crime_corr_enr)


#### Correlation of focus_df (2017-2025)

In [ ]:
focus_df_agg = aggregate(focus_df)

cols = [
    'Crime_Count',  # Temporal
    'Avg_Hour', 'Weekend_Ratio',  # Time patterns
    'Population', 'Mode_Hour', 'Pop_Density', 'Area_sqkm',  # Demographic/spatial
     'Avg_Num_Stations_1km',  # Police presence
    'Avg_Victims', 'Avg_Suspects',  # Crime characteristics
    'Crime_Rate_per_1000', 'Crime_Density_sqkm',  # Derived metrics
]
corr_matrix = focus_df_agg[cols].corr(numeric_only=True)

plt.figure(figsize=(15, 12))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix of 2017-2024 Dataset Features', fontsize=16)
plt.show()

# Focus on Crime_Count correlations
crime_corr = corr_matrix['Crime_Count'].sort_values(ascending=False)
print("\nTop Predictors for Crime_Count:\n", crime_corr)

#### Correlation of train_df (2017-2024)

In [ ]:
cols = [
    'Crime_Count',  # Temporal
    'Avg_Hour', 'Weekend_Ratio',  # Time patterns
    'Population', 'Mode_Hour', 'Pop_Density', 'Area_sqkm',  # Demographic/spatial
     'Avg_Num_Stations_1km',  # Police presence
    'Avg_Victims', 'Avg_Suspects',  # Crime characteristics
    'Crime_Rate_per_1000', 'Crime_Density_sqkm',  # Derived metrics
]

print(train_df.shape)

corr_matrix = train_df[cols].corr(numeric_only=True)

plt.figure(figsize=(15, 12))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix of 2017-2024 Dataset Features', fontsize=16)
plt.show()

# Focus on Crime_Count correlations
crime_corr = corr_matrix['Crime_Count'].sort_values(ascending=False)
print("\nTop Predictors for Crime_Count:\n", crime_corr)

#### Correlation 2017-2024 focus crimes with selected features

In [ ]:
cols = ['Crime_Count',      # Two months ago
    'Population',              # Demographic
    'Area_sqkm',               # Spatial
    'Avg_Num_Stations_1km',    # Police presence
    'Weekend_Ratio',           # Temporal pattern
    'Avg_Hour',                # Time of day
    'Avg_Victims',             # Crime severity
    'Avg_Suspects'             # Crime characteristics
]

print(train_df.shape)

corr_matrix = train_df[cols].corr(numeric_only=True)

plt.figure(figsize=(15, 12))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix of 2017-2024 Dataset & Selected Features', fontsize=16)
plt.show()

# Focus on Crime_Count correlations
crime_corr = corr_matrix['Crime_Count'].sort_values(ascending=False)
print("\nTop Predictors for Crime_Count:\n", crime_corr)

## Feature selection

In [ ]:
selected_features = [       # Two months ago
    'Population',              # Demographic
    'Area_sqkm',               # Spatial
    'Avg_Num_Stations_1km',    # Police presence
    'Weekend_Ratio',           # Temporal pattern
    'Avg_Hour',                # Time of day
    'Avg_Victims',             # Crime severity
    'Avg_Suspects'             # Crime characteristics
]

rand_seed = 42
cv = 4


## XGBoost Classifier

In [ ]:
xgb_df = train_df.copy()

xgb_df.shape

#### With class imbalance

In [ ]:
# --------- FEATURE SELECTION--------

XGB_X = xgb_df[selected_features]
XGB_y = xgb_df['Alarm_Level']

XGB_y = XGB_y.map({'Low': 0, 'Medium': 1, 'High': 2})

# ------ TRAIN AND TEST SPLIT--------

size = 0.25

XGB_X_train, XGB_X_test, XGB_y_train, XGB_y_test = train_test_split(XGB_X, XGB_y, test_size=size, random_state=rand_seed, stratify=XGB_y)

# --------- MODEL TRAINING & PARAMETERS SELECTION -----------------

param_grid = {
# Controls the complexity of the trees
'max_depth': [3, 4, 5, 6, 7],

# Controls the step size. Smaller values require more trees.
 'learning_rate': [0.01, 0.05, 0.1],

# Number of boosting rounds.
'n_estimators': [100, 200, 300, 400],

# Regularization parameters to prevent overfitting
'gamma': [0, 0.1, 0.5], # Minimum loss reduction to make a split
'subsample': [0.7, 0.8, 0.9], # Fraction of training data to use per tree
'colsample_bytree': [0.7, 0.8, 0.9], # Fraction of features to use per tree

# L1 and L2 regularization
'reg_alpha': [0, 0.01, 0.1], # L1 regularization
'reg_lambda': [1, 1.5, 2] # L2 regularization
}

xgb_model = xgb.XGBClassifier(random_state=rand_seed)

xgb_grid_search = RandomizedSearchCV(xgb_model, param_grid, cv=cv, scoring="accuracy", n_jobs=-1, random_state=rand_seed)
xgb_grid_search.fit(XGB_X_train, XGB_y_train)

best_xgb = xgb_grid_search.best_estimator_

xgb_y_pred = best_xgb.predict(XGB_X_test)

# ----------------RESULTS------------------

print(f"\n--- XGBoost Classifier with {size*100}% Test Size Results ---")
print("Best Parameters:\n", xgb_grid_search.best_params_)
test_accuracy = accuracy_score(XGB_y_test, xgb_y_pred)

print(f"\nTest Accuracy: {test_accuracy:.4f}")
print('CV mean:', xgb_grid_search.best_score_)
print('The fit score:', best_xgb.score(XGB_X_train, XGB_y_train))

# Determine if gap is underfit, overfit, or good fit
gap_score = best_xgb.score(XGB_X_train, XGB_y_train) - test_accuracy

if gap_score < 0.05:
    gap = 'Good Fit'
elif gap_score >= 0.05 and gap_score < 0.15:
    gap = 'Overfit'
else:
    gap = 'Underfit'

print('Underfit or Overfit?:', gap_score, gap)
print("\nClassification Report:\n", classification_report(XGB_y_test, xgb_y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(XGB_y_test, xgb_y_pred))


In [ ]:
# --------- FEATURE SELECTION--------

XGB_X = xgb_df[selected_features]
XGB_y = xgb_df['Alarm_Level']

XGB_y = XGB_y.map({'Low': 0, 'Medium': 1, 'High': 2})

# ------ TRAIN AND TEST SPLIT--------

size = 0.25

XGB_X_train, XGB_X_test, XGB_y_train, XGB_y_test = train_test_split(XGB_X, XGB_y, test_size=size, random_state=rand_seed, stratify=XGB_y)

# --------- MODEL TRAINING & PARAMETERS SELECTION -----------------
#{'subsample': 0.9, 'reg_lambda': 1.5, 'reg_alpha': 0.1,
# 'n_estimators': 400, 'max_depth': 5, 'learning_rate': 0.05, 'gamma': 0, 'colsample_bytree': 0.7
param_grid = {
# Controls the complexity of the trees
'max_depth': [5],

# Controls the step size. Smaller values require more trees.
 'learning_rate': [0.05],

# Number of boosting rounds.
'n_estimators': [400],

# Regularization parameters to prevent overfitting
'gamma': [0], # Minimum loss reduction to make a split
'subsample': [0.9], # Fraction of training data to use per tree
'colsample_bytree': [0.7], # Fraction of features to use per tree

# L1 and L2 regularization
'reg_alpha': [0.01], # L1 regularization
'reg_lambda': [1.5] # L2 regularization
}

xgb_model = xgb.XGBClassifier(random_state=rand_seed)

xgb_grid_search = RandomizedSearchCV(xgb_model, param_grid, cv=cv, scoring="accuracy", n_jobs=-1, random_state=rand_seed)
xgb_grid_search.fit(XGB_X_train, XGB_y_train)

best_xgb = xgb_grid_search.best_estimator_

xgb_y_pred = best_xgb.predict(XGB_X_test)

# ----------------RESULTS------------------

print(f"\n--- XGBoost Classifier with {size*100}% Test Size Results ---")
print("Best Parameters:\n", xgb_grid_search.best_params_)
test_accuracy = accuracy_score(XGB_y_test, xgb_y_pred)

print(f"\nTest Accuracy: {test_accuracy:.4f}")
print('CV mean:', xgb_grid_search.best_score_)
print('The fit score:', best_xgb.score(XGB_X_train, XGB_y_train))

# Determine if gap is underfit, overfit, or good fit
gap_score = best_xgb.score(XGB_X_train, XGB_y_train) - test_accuracy

if gap_score < 0.05:
    gap = 'Good Fit'
elif gap_score >= 0.05 and gap_score < 0.15:
    gap = 'Overfit'
else:
    gap = 'Underfit'

print('Underfit or Overfit?:', gap_score, gap)
print("\nClassification Report:\n", classification_report(XGB_y_test, xgb_y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(XGB_y_test, xgb_y_pred))


#### Test prediction on 2025 data using trained 2017-2024 XGBoost Model

In [ ]:
X_2025 = test_df[selected_features]

# map true labels to numeric once (Series)
y_2025 = test_df['Alarm_Level'].map({'Low': 0, 'Medium': 1, 'High': 2})

y_2025_pred = best_xgb.predict(X_2025)

print("\n2025 Classification Report:\n", classification_report(y_2025, y_2025_pred))

# Create results dataframe and attach numeric predictions
test_df_results = test_df.copy()
test_df_results['Predicted_Alarm_Level'] = y_2025_pred

# Map numeric predictions back to label strings
test_df_results['Predicted_Alarm_Level'] = test_df_results['Predicted_Alarm_Level'].map({0: 'Low', 1: 'Medium', 2: 'High'})

print("\n2025 Barangay-Month Predictions:\n", test_df_results[['Barangay', 'Month', 'Weekday', 'Time_of_Day', 'Crime_Count', 'Alarm_Level', 'Predicted_Alarm_Level']])

# show counts
print(test_df_results[['Crime_Count', 'Alarm_Level']].value_counts())

#test_df_results.to_csv('test_2025_results.csv', index=False)
